# Tool Calling From Scratch — S01 companion / optional extension

This optional notebook builds the mechanism underneath the main S01 lab one layer at a time. Start here when you want to understand how an ordinary Python function becomes a model-visible tool and how application code safely dispatches a tool request. Parts 1–4 run locally; Part 6 is an optional live-model comparison.

S01's central principle is: **the model proposes an action; the application validates, authorizes, executes, and observes it.** Here you will implement each responsibility explicitly before relating it to an agent framework.

> [!NOTE]
> This notebook complements `S01_Northstar_Triage_Notebook.ipynb`. Part 3 reuses the same `triage.core.lookup_account` and `triage.core.create_escalation` functions, so the safety behavior remains consistent across both notebooks.

## What you'll be able to do after this notebook

1. Explain why a model's tool request is not itself a function execution.
2. Build a tool registry and a generic dispatcher from scratch.
3. Trace a complete, bounded tool-calling loop end to end.
4. Distinguish the model, runtime, tool implementation, and application policy.
5. Reject invalid, unauthorized, and duplicate side-effecting calls.
6. Relate this handmade runtime to the roles played by agent frameworks.

## Progression

| Part | Topic |
|---|---|
| 1 | Function → registry → dispatcher → fake model → bounded loop |
| 2 | Complete weather-notification example |
| 3 | Apply the same pattern to Northstar |
| 4 | Explore safety failure modes |
| 5 | Map the concepts to real frameworks |
| 6 | Optionally replace the fake model with a live model |


---
# Part 1 — Tool Calling Through Familiar Python Concepts

## Step 1: an ordinary Python function

**Concept.** Before any of this touches a model, it's just a function.


In [ ]:
def get_weather(city: str) -> dict:
    '''A synthetic, deterministic stand-in -- no real weather API is called anywhere
    in this notebook.'''
    forecast = {"Seattle": "rainy", "Phoenix": "sunny", "Chicago": "windy"}.get(city, "unknown")
    return {"city": city, "forecast": forecast}

result = get_weather("Seattle")
print(result)


**Observation.** Python called this directly. No model was involved, no "decision" was
made — this is exactly the function you'd write for any other application.

> [!TIP]
> **⚙️ Engineering implication:** every tool a model can ever "use" starts as exactly
> this — an ordinary function you already know how to write, test, and reason about.
> Nothing about calling it from an agent changes what the function itself does.

## Step 2: a tool registry

**Concept.** A registry maps a *name* a model can refer to onto the actual callable.
This is how a framework "registers" a tool and finds it later by name alone.


In [ ]:
tools = {
    "weather": get_weather,
}

# A model will eventually refer to "weather" by name -- never by Python object
# reference, since the model never sees Python objects, only text/JSON.
print(list(tools.keys()))
print(tools["weather"]("Phoenix"))


**Observation.** Looking a function up by its string name and calling it is the entire
mechanism — `tools["weather"](...)` is not a simplification of what a framework does,
it genuinely is most of what a framework's tool registry does.

## Step 3: a generic dispatcher

**Concept.** A dispatcher is the piece of code that actually does: look up the tool,
check the arguments are the right shape, check the caller is allowed to run it, execute
it, and hand back a controlled result — never letting a model-supplied name or argument
reach a real function unchecked.

> [!IMPORTANT]
> `dispatch()` is **this notebook's name for an illustrative pattern** — real frameworks
> use different names and class shapes (Part 5 maps this precisely). The mechanism
> (lookup → validate → authorize → execute → observe) is what's universal, not the name.


In [ ]:
class AgentRuntime:
    '''A small, from-scratch illustration of what an agent framework's tool-execution
    layer actually does. Real frameworks add more (async execution, streaming,
    distributed state) -- the core mechanism below is not a simplification of theirs,
    it is the same mechanism.
    '''
    def __init__(self, tools: dict, *, authorize=None, side_effecting: frozenset = frozenset()):
        self.tools = tools
        # authorize(name, args, state) -> bool. Defaults to "always allowed" -- a real
        # application almost never wants this default; Part 3 replaces it.
        self.authorize = authorize or (lambda name, args, state: True)
        # Tool names whose repeat execution must be judged by a BUSINESS key, not by
        # whether the arguments happen to be byte-for-byte identical to a prior call.
        self.side_effecting = side_effecting
        self._seen_calls = set()
        self._completed_side_effects = set()

    def dispatch(self, name: str, args: dict, state: dict, *, dedup_key=None) -> dict:
        # 1. Lookup -- an unknown tool name is rejected before anything else happens.
        if name not in self.tools:
            return {"error": f"unknown tool: {name!r}"}
        # 2. Validation -- the shape of the arguments, not their business meaning yet.
        if not isinstance(args, dict):
            return {"error": "tool arguments must be a dict"}
        fingerprint = f"{name}:{sorted(args.items())}"
        if fingerprint in self._seen_calls:
            return {"error": f"repeated tool request: {name}"}
        # 3. Duplicate-side-effect check -- keyed on the BUSINESS entity, not the exact
        #    call, so a second request with different wording still gets caught.
        if name in self.side_effecting:
            key = dedup_key if dedup_key is not None else fingerprint
            if key in self._completed_side_effects:
                return {"error": f"duplicate side effect blocked: {name}"}
        # 4. Authorization -- a separate question from "is this well-formed."
        if not self.authorize(name, args, state):
            return {"error": f"not authorized: {name}"}
        self._seen_calls.add(fingerprint)
        # 5. Execution -- the ONLY point any real side effect happens.
        try:
            outcome = self.tools[name](**args)
        except Exception as exc:
            return {"error": f"{type(exc).__name__}: {exc}"}
        # 6. Observe -- record success so future duplicate checks can see it.
        if name in self.side_effecting:
            self._completed_side_effects.add(dedup_key if dedup_key is not None else fingerprint)
        return {"result": outcome}


In [ ]:
runtime = AgentRuntime(tools)
print(runtime.dispatch("weather", {"city": "Seattle"}, state={}))
print(runtime.dispatch("send_email", {"to": "x"}, state={}))          # unknown tool
print(runtime.dispatch("weather", "not-a-dict", state={}))             # invalid arguments


**Observation.** Three calls, three different outcomes, none of which required touching
`get_weather` itself — the dispatcher's job is entirely about *deciding whether to call
it*, not about weather.

> [!IMPORTANT]
> **⚙️ Engineering implication:** notice that nothing above required a model at all.
> Validation, authorization, and execution-control are pure application-engineering
> concerns — you'd want all of this even if a human, not a model, were requesting tools.
> The model only changes *what proposes* the request; it never changes *who's allowed
> to act on it*.

## Step 4: a simulated LLM decision

**Concept.** A real model call returns a structured proposal. Before using a real one
(Part 6, optional), use a deterministic stand-in so the loop's *mechanism* is
observable and repeatable.

> [!CAUTION]
> `fake_llm_step` below is **simulated LLM behavior** — a hand-written function that
> returns the same shape a real model's tool-call response would, with none of a real
> model's variability. It is not a call to any model API.


In [ ]:
def fake_llm_step(user_request: str, state: dict) -> dict:
    '''SIMULATED model behavior -- deterministic, not a real API call.
    Returns either {"tool": name, "args": {...}} or {"final": "..."}.
    '''
    history = state.get("history", [])
    if not history:
        return {"tool": "weather", "args": {"city": "Seattle"}}
    last_result = history[-1]["outcome"].get("result")
    forecast = (last_result or {}).get("forecast")
    return {"final": f"The forecast for Seattle is {forecast}."}

print(fake_llm_step("What's the weather in Seattle?", {"history": []}))


## Step 5: a bounded agent loop

**Concept.** Now connect every piece: the (simulated) model proposes, the runtime
dispatches, the result becomes an observation, and the model gets another turn — until
it gives a final answer or a configured step limit is reached. **Never an uncontrolled
`while True`.**


In [ ]:
def run_loop(user_request: str, runtime: AgentRuntime, llm_step, *,
             max_steps: int = 5, initial_state: dict | None = None) -> dict:
    state = {"history": [], **(initial_state or {})}
    for step in range(1, max_steps + 1):
        decision = llm_step(user_request, state)
        if "final" in decision:
            return {"status": "done", "response": decision["final"], "history": state["history"]}
        name, args = decision.get("tool"), decision.get("args", {})
        outcome = runtime.dispatch(name, args, state, dedup_key=decision.get("dedup_key"))
        state["history"].append({"step": step, "tool": name, "args": args, "outcome": outcome})
        if "error" in outcome:
            # Tool-dispatch failures stop the loop -- they do not get silently retried.
            # (A *verification* failure on a final answer, covered in the existing S01
            # material, is a different, bounded-retry case -- not this one.)
            return {"status": "stopped", "reason": outcome["error"], "history": state["history"]}
    return {"status": "stopped", "reason": "max_steps exceeded", "history": state["history"]}


outcome = run_loop("What's the weather in Seattle?", AgentRuntime(tools), fake_llm_step)
for entry in outcome["history"]:
    print(entry)
print(outcome["status"], "->", outcome.get("response") or outcome.get("reason"))


**Observation.** One tool call, one final answer, a fully inspectable history — and the
loop would have stopped cleanly at `max_steps` even if `fake_llm_step` never produced a
`"final"` key.

> [!IMPORTANT]
> **⚙️ Engineering implication — the central principle, now literally true of the code
> above:** `fake_llm_step` **proposes**; `run_loop` and `AgentRuntime.dispatch`
> **validate, authorize, execute, and observe**. Nothing in `fake_llm_step` ever touched
> `get_weather` directly — it couldn't have, even if it tried.

### ✅ Checkpoint

If `fake_llm_step` returned `{"tool": "weather", "args": {"city": "Seattle"}}` on every
single call, forever, what would actually stop this loop from running forever?

<details>
<summary>Answer</summary>

`max_steps` in `run_loop` — nothing about the model's behavior bounds the loop; the
application-owned step limit does. This is true even though, in this specific case,
`AgentRuntime`'s repeated-call detection would *also* stop it one step sooner (the exact
same `{"city": "Seattle"}` call would be flagged as a repeated request) — but that's a
second, independent safeguard, not the only one.

</details>


---
# Part 2 — A Complete Weather Notification Example

**Customer request:** *"Check tomorrow's weather in Seattle and notify me if rain is
expected."*


In [ ]:
def get_weather_for_date(city: str, date: str) -> dict:
    # Deterministic synthetic data -- no real weather API or account is used.
    forecasts = {("Seattle", "tomorrow"): "rainy", ("Phoenix", "tomorrow"): "sunny"}
    return {"city": city, "date": date, "forecast": forecasts.get((city, date), "unknown")}

_sent_notifications = []

def send_notification(message: str) -> dict:
    # A mock notification "service" -- appends to a list instead of sending anything real.
    _sent_notifications.append(message)
    return {"sent": True, "message": message}

weather_tools = {"get_weather": get_weather_for_date, "send_notification": send_notification}


**Simulated model logic for this scenario:** request the weather, decide whether rain
was forecast, request a notification only if it was, then give a final answer.


In [ ]:
def weather_notification_llm_step(user_request: str, state: dict) -> dict:
    history = state.get("history", [])
    if not history:
        return {"tool": "get_weather", "args": {"city": "Seattle", "date": "tomorrow"}}
    last = history[-1]
    if last["tool"] == "get_weather":
        forecast = last["outcome"].get("result", {}).get("forecast")
        if forecast == "rainy":
            return {"tool": "send_notification",
                    "args": {"message": "Rain is expected tomorrow in Seattle -- bring an umbrella."},
                    "dedup_key": "rain-notification-seattle-tomorrow"}
        return {"final": f"Checked tomorrow's Seattle forecast: {forecast}. No notification needed."}
    return {"final": "Notification sent -- rain is expected tomorrow in Seattle."}


runtime = AgentRuntime(weather_tools, side_effecting=frozenset({"send_notification"}))
result = run_loop("Check tomorrow's weather in Seattle and notify me if rain is expected.",
                   runtime, weather_notification_llm_step)
for entry in result["history"]:
    print(entry)
print(result["status"], "->", result.get("response"))
print("Notifications actually sent:", _sent_notifications)


**Observation.** Weather → rainy → notification sent, in two tool calls plus a final
answer — and `_sent_notifications` proves the side effect actually happened, not just
that the loop claimed it did.

## The no-rain scenario — the notification tool is never called

Same tools, same loop, a different city.


In [ ]:
def sunny_llm_step(user_request: str, state: dict) -> dict:
    history = state.get("history", [])
    if not history:
        return {"tool": "get_weather", "args": {"city": "Phoenix", "date": "tomorrow"}}
    forecast = history[-1]["outcome"].get("result", {}).get("forecast")
    return {"final": f"Checked tomorrow's Phoenix forecast: {forecast}. No notification needed."}


_sent_notifications.clear()
runtime2 = AgentRuntime(weather_tools, side_effecting=frozenset({"send_notification"}))
result2 = run_loop("Check tomorrow's weather in Phoenix and notify me if rain is expected.",
                    runtime2, sunny_llm_step)
for entry in result2["history"]:
    print(entry)
print(result2["status"], "->", result2.get("response"))
print("Notifications actually sent:", _sent_notifications)
assert _sent_notifications == [], "no-rain scenario must never call send_notification"


**Observation.** `_sent_notifications` stays empty — confirmed, not assumed. The
decision *not* to call a tool is as much a part of correct agent behavior as deciding
to call one; a framework that always reaches for every available tool regardless of
need is not more capable, just noisier and more expensive (S03's cost lessons apply
here directly).

### ✍️ Learner exercise

Add a third city/date combination with forecast `"snowy"`, and extend
`weather_notification_llm_step` so it also notifies on snow, reusing the same
`dedup_key` pattern. Confirm by running it that exactly one notification is sent, not
zero and not two.

<details>
<summary>Instructor solution</summary>

Add `("Chicago", "tomorrow"): "snowy"` to the `forecasts` dict, change the condition to
`if forecast in ("rainy", "snowy")`, and reuse a dedup key like
`f"weather-notification-{city}-{date}"` computed from the actual city/date rather than a
hardcoded string — the hardcoded `"rain-notification-seattle-tomorrow"` key above only
works because this example never varies the city; a general solution must key on the
actual request, not a literal string.

</details>


---
# Part 3 — Applying the Same Pattern to Northstar

**Customer requirement:** *"Review ticket T-104. Check the customer account. If the customer has premium support and the issue is critical, escalate the ticket."*

This section reuses `triage.core.lookup_account` and `triage.core.create_escalation` directly and unmodified. You are not learning a second implementation; you are viewing the same Northstar safety rules through the `AgentRuntime` vocabulary built in this notebook.


In [ ]:
import json
from pathlib import Path
from triage.core import lookup_account, create_escalation

TICKETS = {t["ticket_id"]: t for t in json.loads(Path("data/tickets.json").read_text())}

def get_ticket(ticket_id: str) -> dict:
    '''A small, notebook-local read tool -- not part of triage/core.py, since the
    existing lab passes the ticket in directly. Added here so Northstar gets the same
    three-tool shape (get/lookup/act) as the weather example.'''
    ticket = TICKETS.get(ticket_id)
    if ticket is None:
        raise KeyError(f"no such ticket: {ticket_id}")
    return ticket


## Registering Northstar's real tools

`create_escalation` is **side-effecting** — registered as such, exactly like
`send_notification` was in Part 2.


In [ ]:
def authorize_northstar(name, args, state):
    '''The business-authorization policy for this scenario -- separate from whether the
    arguments are well-formed. This is where 'premium account + high priority' actually
    gets enforced, independent of what the model claims.'''
    if name != "create_escalation":
        return True
    ticket = state.get("ticket", {})
    account_obs = state.get("account_observation")
    is_premium = bool(account_obs and account_obs.get("support_tier") == "premium")
    priority = state.get("priority")
    return (
        args.get("ticket_id") == ticket.get("ticket_id")
        and args.get("account_id") == ticket.get("account_id")
        and priority == "high"
        and is_premium
    )

def northstar_create_escalation(ticket_id, account_id, reason):
    '''Adapts triage.core.create_escalation's trusted-parameter signature to this
    notebook's AgentRuntime, which authorizes separately via `authorize_northstar`
    above rather than passing trust flags as call arguments.'''
    return create_escalation(
        ticket_id, account_id, reason,
        expected_ticket_id=ticket_id, expected_account_id=account_id,
        priority="high", is_premium=True,
    )  # safe: authorize_northstar already confirmed both facts before dispatch ever executes this

def northstar_lookup_account(account_id):
    # NOTE: in a real run, expected_account_id would come from application state, not
    # from the model's own argument -- see the authorization-failure trace below for
    # exactly what goes wrong if you trust the model's account_id unconditionally.
    return lookup_account(account_id, expected_account_id=CURRENT_TICKET["account_id"])

northstar_tools = {
    "get_ticket": get_ticket,
    "lookup_account": northstar_lookup_account,
    "create_escalation": northstar_create_escalation,
}


## A simulated model for the escalation scenario


In [ ]:
CRITICAL_KEYWORDS = ("critical", "urgent", "down", "immediately")

def northstar_llm_step(user_request: str, state: dict) -> dict:
    history = state.get("history", [])
    if not history:
        return {"tool": "get_ticket", "args": {"ticket_id": state["ticket_id"]}}
    last = history[-1]
    if last["tool"] == "get_ticket":
        ticket = last["outcome"]["result"]
        state["ticket"] = ticket
        if ticket.get("account_id"):
            return {"tool": "lookup_account", "args": {"account_id": ticket["account_id"]}}
        return {"final": f"Ticket {ticket['ticket_id']} needs no account lookup; resolved from ticket text alone."}
    if last["tool"] == "lookup_account":
        account = last["outcome"]["result"]
        state["account_observation"] = account
        ticket = state["ticket"]
        msg = ticket["message"].lower()
        priority = "high" if any(w in msg for w in CRITICAL_KEYWORDS) else "medium"
        state["priority"] = priority
        is_premium = account.get("support_tier") == "premium"
        if priority == "high" and is_premium:
            return {"tool": "create_escalation",
                    "args": {"ticket_id": ticket["ticket_id"], "account_id": ticket["account_id"],
                              "reason": f"High-priority issue on a premium account: {ticket['message'][:60]}"},
                    "dedup_key": ticket["ticket_id"]}
        return {"final": f"Ticket {ticket['ticket_id']}: priority={priority}, "
                          f"premium={is_premium} -- no auto-escalation; flag for human review."}
    return {"final": f"Ticket {state['ticket']['ticket_id']} escalated successfully."}


In [ ]:
CURRENT_TICKET = TICKETS["T-110"]  # premium account, critical wording -- escalation expected
runtime3 = AgentRuntime(northstar_tools, authorize=authorize_northstar,
                        side_effecting=frozenset({"create_escalation"}))
result3 = run_loop("Review ticket T-110.", runtime3, northstar_llm_step,
                    initial_state={"ticket_id": "T-110"})
for entry in result3["history"]:
    print(entry)
print(result3["status"], "->", result3.get("response") or result3.get("reason"))


**Observation.** `get_ticket` → `lookup_account` → `create_escalation` → final answer,
reusing `triage.core`'s real `create_escalation` for the actual write, gated by
`authorize_northstar`'s policy — the exact same "premium + high priority" rule
`triage/core.py`'s own dispatcher enforces, expressed through this notebook's generic
`AgentRuntime` instead.

## The existing Northstar authorization-failure example, replayed here

S01's handout poses this exact trace as a hands-on challenge. Reproduce it with the
`AgentRuntime` you just built, to see the same failure in the vocabulary of *this*
notebook:

```text
1. Ticket T-104 arrives for account A-204.
2. Model requests lookup for account A-999 (not the ticket's own account).
3. Application must deny this BEFORE any tool executes.
```


In [ ]:
CURRENT_TICKET = TICKETS["T-104"]  # account A-204

def misbehaving_llm_step(user_request, state):
    # Deliberately requests the WRONG account -- this is what a prompt-injected or
    # simply buggy model proposing an out-of-scope lookup looks like.
    return {"tool": "lookup_account", "args": {"account_id": "A-999"}}

runtime4 = AgentRuntime(northstar_tools, authorize=authorize_northstar)
result4 = run_loop("Review ticket T-104.", runtime4, misbehaving_llm_step)
for entry in result4["history"]:
    print(entry)
print(result4["status"], "->", result4.get("reason"))
assert result4["status"] == "stopped"
assert "PermissionError" in result4["reason"]


**Observation.** The dispatcher denies the lookup **before** any tool body runs —
`northstar_lookup_account`'s own `expected_account_id` check (reusing
`triage.core.lookup_account` unmodified) catches the mismatch. No account data for
`A-999` is ever read, let alone returned.

> [!IMPORTANT]
> **⚙️ Engineering implication:** correcting a later step can never undo an earlier
> unauthorized action — which is exactly why this check has to happen **before**
> execution, not be caught afterward by inspecting the final answer.


---
# Part 4 — Framework Safety Responsibilities, Demonstrated Live

Every row below is executed, not just described.

| Failure scenario | Expected behavior |
|---|---|
| Unknown tool requested | Reject the request |
| Invalid tool arguments | Reject with a controlled error |
| Unauthorized account lookup | Deny before execution |
| Tool raises an exception | Return a controlled failure |
| Repeated tool calls | Detect and stop |
| Excessive iterations | Stop at the configured limit |
| Duplicate escalation request | Prevent duplicate side effects |
| Valid arguments but unsafe business action | Enforce separate business authorization |


In [ ]:
demo_runtime = AgentRuntime(northstar_tools, authorize=authorize_northstar,
                            side_effecting=frozenset({"create_escalation"}))
state = {}

print("1. Unknown tool:")
print(" ", demo_runtime.dispatch("delete_database", {}, state))

print("2. Invalid arguments (not a dict):")
print(" ", demo_runtime.dispatch("get_ticket", "T-104", state))

print("3. Unauthorized account lookup (wrong account for this ticket):")
CURRENT_TICKET = TICKETS["T-104"]
print(" ", demo_runtime.dispatch("lookup_account", {"account_id": "A-999"}, state))

print("4. Tool raises an exception (ticket does not exist):")
print(" ", demo_runtime.dispatch("get_ticket", {"ticket_id": "T-does-not-exist"}, state))


In [ ]:
print("5. Repeated identical tool call:")
fresh_runtime = AgentRuntime(northstar_tools, authorize=authorize_northstar)
s = {}
print(" first: ", fresh_runtime.dispatch("get_ticket", {"ticket_id": "T-104"}, s))
print(" repeat:", fresh_runtime.dispatch("get_ticket", {"ticket_id": "T-104"}, s))

print("\n6. Excessive iterations (a model that never stops proposing tools):")
# Args must genuinely vary each call -- a tool requested with IDENTICAL arguments
# every time would trip the repeated-call check first (scenario 5), not this one.
_counter = {"n": 0}
def never_stops(user_request, state):
    _counter["n"] += 1
    return {"tool": "noop", "args": {"n": _counter["n"]}}
r = run_loop("...", AgentRuntime({"noop": lambda n: {"ok": True, "n": n}}), never_stops, max_steps=3)
print(" ", r["status"], r["reason"], "-- stopped after", len(r["history"]), "steps, not forever")
assert r["reason"] == "max_steps exceeded"


In [ ]:
print("7. Duplicate escalation request (different wording, same ticket):")
CURRENT_TICKET = TICKETS["T-110"]
state7 = {"ticket": CURRENT_TICKET, "account_observation": {"support_tier": "premium"}, "priority": "high"}
dup_runtime = AgentRuntime(northstar_tools, authorize=authorize_northstar,
                           side_effecting=frozenset({"create_escalation"}))
first = dup_runtime.dispatch("create_escalation",
    {"ticket_id": "T-110", "account_id": "A-417", "reason": "first wording"}, state7, dedup_key="T-110")
second = dup_runtime.dispatch("create_escalation",
    {"ticket_id": "T-110", "account_id": "A-417", "reason": "a totally different second wording"}, state7, dedup_key="T-110")
print(" first call: ", first)
print(" second call:", second)
assert "result" in first and "error" in second

print("\n8. Valid arguments, but unsafe business action (standard tier, not premium):")
state8 = {"ticket": TICKETS["T-111"], "account_observation": {"support_tier": "standard"}, "priority": "high"}
print(" ", dup_runtime.dispatch("create_escalation",
    {"ticket_id": "T-111", "account_id": "A-310", "reason": "well-formed, but not entitled to escalate"}, state8, dedup_key="T-111"))


**Observation.** All eight rows produce the controlled, expected outcome — none of them
required a real model to demonstrate, because every one of them is an **application**
responsibility, not a model-behavior question.

> [!IMPORTANT]
> **The single sentence this whole notebook has been building toward:** a JSON schema
> tells the model what a well-formed tool call looks like. It does not, and cannot,
> decide who's allowed to make that call, whether this exact call has already happened,
> or whether the business situation actually justifies it. Those are three separate
> questions, answered by three separate pieces of application code — never by the
> schema itself.

### ✍️ Learner exercise

Add a ninth row: a read-only tool (`get_ticket`) versus a side-effecting one
(`create_escalation`) should be allowed different retry policies — reads can often be
retried freely; writes usually shouldn't be, without idempotency. State, in one
sentence, why `AgentRuntime.dispatch`'s repeated-call check and its duplicate-side-effect
check are deliberately two *different* mechanisms rather than one.

<details>
<summary>Instructor solution</summary>

The repeated-call check (`_seen_calls`, keyed on exact arguments) is about **wasted
work** — calling `get_ticket` twice with identical arguments is harmless but
pointless, so it's blocked generically for any tool. The duplicate-side-effect check
(`_completed_side_effects`, keyed on a business `dedup_key`) is about **correctness** —
a second `create_escalation` call with *different* wording would sail past the
exact-argument check and still create a second escalation unless the business-key
check exists separately. Reads only need the first; writes need both.

</details>


---
# Part 5 — Where These Concepts Appear in Real Agent Frameworks

> [!NOTE]
> Every link below was checked against the live documentation/source while this
> notebook was written, including catching one stale URL
> (`python.langchain.com/docs/concepts/tool_calling` now permanently redirects) —
> verify them again yourself if you're reading this much later; frameworks move fast.

| This notebook | LangChain (the model-facing layer) | LangGraph (the execution layer) |
|---|---|---|
| `tools = {"weather": get_weather}` (registry) | A plain Python function passed to `tools=[...]` | Tools registered with a prebuilt or custom graph node |
| Tool schema (implicit: function signature) | Inferred from the function's type hints/docstring, or an explicit JSON schema | Same schemas, consumed by the node |
| `fake_llm_step` returning `{"tool": ..., "args": ...}` | `model.bind_tools([...])`; the response exposes requests via `response.tool_calls` (`name`, `args`, `id`) | N/A — LangGraph consumes the same `AIMessage.tool_calls` |
| `AgentRuntime.dispatch()` | The hand-written loop the docs show for using tools *outside* an agent: look up by `name`, call `tool.invoke(call)` | `ToolNode` — a prebuilt graph node that runs the tool calls in the latest AI message |
| `state` dict threaded through `run_loop` | The growing message list (`messages`) | LangGraph's typed graph state |
| `outcome["result"]` as an observation | A `ToolMessage`, matched back to its request by `tool_call_id` | Same `ToolMessage` convention |
| `run_loop`'s step limit + stop-on-error | Your own loop's exit condition | Graph edges / recursion limits |
| `authorize()` callback | Not a built-in concept — application code you still have to write yourself | `wrap_tool_call` hook, or your own graph node before `ToolNode` |

**Current, verified sources** (fetched live while writing this notebook, not quoted
from memory):

- [LangChain — tool calling and models](https://docs.langchain.com/oss/python/langchain/models) —
  the exact current mechanics: `bind_tools`, `response.tool_calls` fields
  (`name`/`args`/`id`), and the `tool_call_id`-matched `ToolMessage` pattern this
  notebook's `dedup_key`/observation pattern mirrors.
- [LangChain — agents overview](https://docs.langchain.com/oss/python/langchain/agents) —
  the higher-level `create_agent(model=..., tools=[...])` API that hides the loop this
  notebook builds by hand.
- [LangGraph `ToolNode` reference](https://reference.langchain.com/python/langgraph.prebuilt/tool_node/ToolNode) —
  the prebuilt node that plays the role of this notebook's `AgentRuntime.dispatch`.
- [LangGraph `ToolNode` source, `tool_node.py`](https://github.com/langchain-ai/langgraph/blob/main/libs/prebuilt/langgraph/prebuilt/tool_node.py) —
  the actual implementation: tools stored in `self._tools_by_name` (this notebook's
  `self.tools`), unknown-tool and validation errors raised explicitly, and a
  `handle_tool_errors` hook playing the role of this notebook's `except Exception`
  branch.

> [!WARNING]
> Do not treat the mapping above as "LangChain/LangGraph will keep you safe by
> default." `ToolNode`'s `handle_tool_errors` and `wrap_tool_call` hooks are places
> *you* plug in authorization and idempotency policy — a framework gives you the
> mechanism's scaffolding, not a decision about what your business should authorize.
> That part was never going to be handled for you, in this notebook or any framework.

### ✍️ Optional source-exploration exercise (not required)

Open the `tool_node.py` link above. Find the method that resolves a tool call's `name`
to an actual tool object, and the method that actually invokes it. Compare them, in
your own words, to this notebook's `AgentRuntime.dispatch`'s lookup and execution
steps — same mechanism, different names and a lot more production-hardening code
around it (async execution, state/store injection, structured error types).

**No solution is provided for this one** — the point is reading someone else's real,
current source code and recognizing a pattern you already understand, not checking an
answer.


---
# Part 6 — Optional: Replace the Fake LLM with a Real Model Call

> [!CAUTION]
> This optional section makes a billed provider request and requires `OPENAI_API_KEY` and `OPENAI_MODEL`. Skip it if you want to keep the notebook fully local; Parts 1–5 already demonstrate the complete application-controlled tool loop.


In [ ]:
RUN_LIVE = False  # explicit opt-in

if RUN_LIVE:
    import os
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv()  # reads .env from the repo root (or this folder, if present)
    api_key = os.environ.get("OPENAI_API_KEY")
    model_name = os.environ.get("OPENAI_MODEL")
    if not api_key or not model_name:
        raise RuntimeError("Set OPENAI_API_KEY and OPENAI_MODEL before setting RUN_LIVE = True")
    client = OpenAI(api_key=api_key)

    # Explicit tool schemas -- the model only ever sees this description, never the
    # Python implementation.
    tool_schemas = [
        {"type": "function", "function": {
            "name": "get_ticket", "description": "Fetch a Northstar support ticket by ID",
            "parameters": {"type": "object", "properties": {"ticket_id": {"type": "string"}},
                            "required": ["ticket_id"], "additionalProperties": False}}},
        {"type": "function", "function": {
            "name": "lookup_account", "description": "Read-only synthetic account lookup",
            "parameters": {"type": "object", "properties": {"account_id": {"type": "string"}},
                            "required": ["account_id"], "additionalProperties": False}}},
        {"type": "function", "function": {
            "name": "create_escalation", "description": "Escalate a high-priority ticket on a premium account",
            "parameters": {"type": "object", "properties": {
                "ticket_id": {"type": "string"}, "account_id": {"type": "string"}, "reason": {"type": "string"}},
                "required": ["ticket_id", "account_id", "reason"], "additionalProperties": False}}},
    ]

    def real_llm_step(messages: list[dict]) -> dict:
        '''Returns the raw response message so the caller can inspect tool_calls and
        preserve each call's id -- required to send a matching ToolMessage back.'''
        response = client.chat.completions.create(
            model=model_name, messages=messages, tools=tool_schemas, tool_choice="auto", temperature=0,
        )
        return response.choices[0].message

    def run_live_loop(ticket_id: str, *, max_steps: int = 5) -> dict:
        import json as _json
        messages = [
            {"role": "system", "content": "You triage Northstar support tickets using the provided tools only."},
            {"role": "user", "content": f"Review ticket {ticket_id}."},
        ]
        CURRENT_TICKET_LIVE = TICKETS[ticket_id]
        account_observation = None
        priority_live = None
        for step in range(1, max_steps + 1):
            message = real_llm_step(messages)
            if not message.tool_calls:
                return {"status": "done", "response": message.content}
            messages.append({"role": "assistant", "tool_calls": message.tool_calls, "content": message.content or ""})
            for call in message.tool_calls:  # a real response may request several tools at once
                name = call.function.name
                try:
                    args = _json.loads(call.function.arguments)
                except (ValueError, TypeError):
                    args = None
                if name == "create_escalation":
                    is_premium = bool(account_observation and account_observation.get("support_tier") == "premium")
                    authorized = (isinstance(args, dict) and args.get("ticket_id") == ticket_id
                                  and priority_live == "high" and is_premium)
                else:
                    authorized = isinstance(args, dict)
                if not authorized:
                    tool_result = {"error": "not authorized or malformed arguments"}
                elif name == "get_ticket":
                    tool_result = {"result": CURRENT_TICKET_LIVE}
                elif name == "lookup_account":
                    try:
                        tool_result = {"result": northstar_lookup_account(**args)}
                        account_observation = tool_result["result"]
                        msg = CURRENT_TICKET_LIVE["message"].lower()
                        priority_live = "high" if any(w in msg for w in CRITICAL_KEYWORDS) else "medium"
                    except PermissionError as exc:
                        tool_result = {"error": str(exc)}
                elif name == "create_escalation":
                    try:
                        tool_result = {"result": northstar_create_escalation(**args)}
                    except (PermissionError, ValueError) as exc:
                        tool_result = {"error": str(exc)}
                else:
                    tool_result = {"error": f"unknown tool: {name}"}
                # Preserve the tool_call_id -- required so the model can match this
                # result back to the request it made.
                messages.append({"role": "tool", "tool_call_id": call.id, "content": _json.dumps(tool_result)})
        return {"status": "stopped", "reason": "max_steps exceeded"}

    print(run_live_loop("T-110"))
else:
    print("Skipped -- set RUN_LIVE = True (and the required environment variables) to run this live.")


**What this section demonstrates, if you run it:** the exact same five concerns from
Parts 1–4 — validate, authorize, execute, observe, bound the loop — applied to a real
model's tool-call protocol, including the one new, provider-specific requirement:
**preserve each `tool_call_id`** so a result can be matched back to its request, and
**support multiple tool calls in a single response** (the `for call in
message.tool_calls` loop), which the deterministic fake-LLM parts of this notebook
never needed to handle.

---

## 📚 References

- `fde/sessions/S01-verified-agent-loops/handout.md` — the central principle and
  conceptual framing this notebook builds on.
- `fde/sessions/S01-verified-agent-loops/lab/triage/core.py` — the real,
  tested implementation this notebook's Part 3 reuses directly.
- [LangChain — tool calling and models](https://docs.langchain.com/oss/python/langchain/models)
- [LangGraph `ToolNode` reference](https://reference.langchain.com/python/langgraph.prebuilt/tool_node/ToolNode)
- [LangGraph `ToolNode` source](https://github.com/langchain-ai/langgraph/blob/main/libs/prebuilt/langgraph/prebuilt/tool_node.py)

**Execution note:** Parts 1–4 are pure Python, require no model or API key, and were
executed end to end (not just syntax-checked) while this notebook was built. Part 6 was
not executed — it is gated behind `RUN_LIVE` and real credentials by design, and
remains unverified against any live provider.
